<h1 style=\"text-align: center; font-size: 50px;\"> 🤖 MLFlow Registration for Agentic RAG Model</h1>

# Notebook Overview

- Start Execution
- Install and Import Libraries
- Configure Settings
- Define the Agentic RAG Model
- Register the Model to MLFlow
- Log Results to MLFlow

# Start Execution

In [1]:
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("register_model_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [2]:
start_time = time.time()  
logger.info("Notebook execution started.")

2025-12-04 16:40:22 - INFO - Notebook execution started.


# Install and Import Libraries

In [3]:
%%time

%pip install -r ../requirements.txt --quiet 

  DEPRECATION: Setting PIP_CONSTRAINT will not affect build constraints in the future, pip 26.2 will enforce this behaviour change. A possible replacement is to specify build constraints using --build-constraint or PIP_BUILD_CONSTRAINT. To disable this warning without any build constraints set --use-feature=build-constraint or PIP_USE_FEATURE="build-constraint".
Note: you may need to restart the kernel to use updated packages.
CPU times: user 1.29 s, sys: 516 ms, total: 1.81 s
Wall time: 52.7 s


In [4]:
from __future__ import annotations


import tempfile
import shutil
import json
import os
import sys
import warnings
from collections import namedtuple
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, TypedDict

import pandas as pd
import tensorrt_llm

import mlflow.pyfunc
from mlflow.models.signature import ModelSignature
from mlflow.tracking import MlflowClient
from mlflow.types import ColSpec, DataType, Schema

from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langgraph.graph import StateGraph, START, END

from transformers import AutoTokenizer, AutoModelForCausalLM,AutoModel

import torch
import numpy as np
sys.path.append("../src")

# ─────── TRT-LLM ───────
import tensorrt_llm
parent_dir = os.path.dirname(os.path.abspath('.'))
sys.path.append(parent_dir)
from src.trt_llm_langchain import TensorRTLangchain

# Import the new MLflow integration layer
from src.mlflow import Logger

# Import the new MLflow integration layer
from src.mlflow import Logger

from IPython import get_ipython

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[TensorRT-LLM] TensorRT-LLM version: 0.18.0


# Configure Settings

In [5]:
# ------------------------ Suppress Verbose Logs ------------------------
warnings.filterwarnings("ignore")

In [6]:
# ------------------------- MLflow Experiment Configuration -------------------------
MODEL_NAME = "Agentic_RAG_Model"
RUN_NAME = f"Register_{MODEL_NAME}_Run"
EXPERIMENT_NAME = "Agentic_RAG_Experiment"
DEMO_PATH = "../demo"

# Load configuration
import yaml
with open("../configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Define the Agentic RAG Model

In [7]:
# The complex RagAgenticModel class has been extracted to src/mlflow/model.py
# Registration is now handled by the generic Logger class

# Get model path from config
model_path = config.get("model_path")
NEMOTRON_DIR = model_path  # Use config-based path

print(f"Using model path from config: {model_path}")
print("RagAgenticModel business logic has been extracted to src/mlflow/model.py")

Using model path from config: nvidia/Llama-3.1-Nemotron-Nano-8B-v1
RagAgenticModel business logic has been extracted to src/mlflow/model.py


# Register the Model to MLFlow

In [8]:
# 1. Set MLflow tracking URI and experiment
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "/phoenix/mlflow"))
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)
print(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")

Using MLflow tracking URI: /phoenix/mlflow
Experiment: Agentic_RAG_Experiment


In [9]:
# Create MLflow signature for the model
input_schema = Schema([
    ColSpec(DataType.string, name="query")
])

output_schema = Schema([
    ColSpec(DataType.string, name="answer"),
    ColSpec(DataType.string, name="retrieved_chunks"),  # Will be serialized as JSON
    ColSpec(DataType.string, name="messages")           # Will be serialized as JSON
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)
print("Created MLflow signature for model")

Created MLflow signature for model


In [10]:
# ------------------------- Helper Functions -------------------------

def create_tensorrt_models_dict():
    """Create TensorRT-LLM models dictionary from config (includes Nemotron)."""
    nemotron_path = config.get("model_path")  # This is the Nemotron/TensorRT-LLM model
    return {
        "nemotron_tensorrt_llm": nemotron_path,  # Nemotron Nano 8B with TensorRT-LLM
        "embedding_model": config.get("embedding_model", "sentence-transformers/all-MiniLM-L6-v2")
    }

def load_models_for_onnx_conversion():
    """
    Load all models into memory for ONNX conversion.
    Includes Nemotron-based TensorRT-LLM model.
    
    Returns:
        tuple: (tensorrt_models_dict, loaded_models_dict)
    """
    import torch
    from transformers import AutoTokenizer, AutoModel
    
    tensorrt_models = create_tensorrt_models_dict()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Load models into memory
    loaded_models = {}
    
    # Load Nemotron TensorRT-LLM model if path exists
    nemotron_path = tensorrt_models["nemotron_tensorrt_llm"]
    if nemotron_path:
        try:
            # Check if it's a local directory or HuggingFace repo
            if os.path.exists(nemotron_path):
                # Local TensorRT engine
                loaded_models['nemotron_trt_llm'] = TensorRTLangchain(
                    model_path=nemotron_path,
                    temperature=config.get("temperature", 0.7),
                    max_tokens=config.get("max_tokens", 512)
                )
                logger.info(f"Loaded Nemotron TensorRT-LLM from local: {nemotron_path}")
            else:
                # HuggingFace repo ID (e.g., nvidia/Llama-3.1-Nemotron-Nano-8B-v1)
                logger.info(f"Nemotron model configured as HuggingFace repo: {nemotron_path}")
                logger.info("Model will be downloaded on-demand during inference")
        except Exception as e:
            logger.warning(f"Could not load Nemotron TensorRT-LLM model: {e}")
    
    # Load embedding model (both the wrapper and raw model for ONNX export)
    try:
        embedding_model_name = tensorrt_models["embedding_model"]
        loaded_models['embedding_model'] = HuggingFaceEmbeddings(
            model_name=embedding_model_name
        )
        # Also load the raw transformer model for ONNX conversion
        loaded_models['embedding_model_raw'] = AutoModel.from_pretrained(embedding_model_name)
        logger.info(f"Loaded embedding model: {embedding_model_name}")
    except Exception as e:
        logger.warning(f"Could not load embedding model: {e}")
    
    logger.info("All models loaded into memory for ONNX conversion")
    return tensorrt_models, loaded_models

def create_and_convert_onnx_models(loaded_models):
    """
    Create ONNX versions of models and save them to a temporary directory.
    Note: Nemotron TensorRT-LLM models are kept in native engine format.
    
    Args:
        loaded_models: Dictionary of loaded model objects
    
    Returns:
        str: Path to directory containing ONNX model files
    """
    import torch
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Create temp directory for ONNX models
    onnx_dir = os.path.join(tempfile.gettempdir(), "onnx_models_agentic_rag")
    if os.path.exists(onnx_dir):
        shutil.rmtree(onnx_dir)
    os.makedirs(onnx_dir)
    
    try:
        # Convert embedding model to ONNX if available
        if 'embedding_model_raw' in loaded_models:
            embedding_onnx_path = os.path.join(onnx_dir, "embedding_model.onnx")
            
            # Get the raw transformer model
            raw_model = loaded_models['embedding_model_raw']
            raw_model.eval()  # Set to evaluation mode
            
            # Create dummy input with proper shape
            dummy_input_ids = torch.randint(0, 1000, (1, 128), dtype=torch.long)
            dummy_attention_mask = torch.ones((1, 128), dtype=torch.long)
            
            # Export to ONNX with proper input handling
            torch.onnx.export(
                raw_model,
                (dummy_input_ids, dummy_attention_mask),
                embedding_onnx_path,
                export_params=True,
                opset_version=14,
                do_constant_folding=True,
                input_names=['input_ids', 'attention_mask'],
                output_names=['last_hidden_state', 'pooler_output'],
                dynamic_axes={
                    'input_ids': {0: 'batch_size', 1: 'sequence'},
                    'attention_mask': {0: 'batch_size', 1: 'sequence'},
                    'last_hidden_state': {0: 'batch_size', 1: 'sequence'},
                    'pooler_output': {0: 'batch_size'}
                }
            )
            logger.info(f"✅ Converted embedding model to ONNX: {embedding_onnx_path}")
        else:
            logger.warning("Embedding model not available for ONNX conversion")
        
        # For Nemotron TensorRT-LLM, we keep the native engine format (not converting to ONNX)
        # as TensorRT already provides optimized inference for Nemotron models
        if 'nemotron_trt_llm' in loaded_models:
            logger.info("✅ Nemotron TensorRT-LLM model kept in native engine format (optimized)")
            
        logger.info(f"📁 ONNX models saved to directory: {onnx_dir}")
        return onnx_dir
        
    except Exception as e:
        logger.error(f"❌ Error during ONNX conversion: {str(e)}")
        logger.info("⚠️ Continuing without ONNX conversion...")
        import traceback
        logger.debug(traceback.format_exc())
        return onnx_dir

def prepare_models_with_onnx(tensorrt_models, onnx_dir):
    """
    Prepare models directory containing both Nemotron TensorRT-LLM and ONNX files.
    
    Args:
        tensorrt_models: Dictionary of TensorRT model paths (includes Nemotron)
        onnx_dir: Directory containing ONNX model files
    
    Returns:
        str: Path to directory containing all model files
    """
    models_dir = os.path.join(tempfile.gettempdir(), "all_agentic_rag_models")
    if os.path.exists(models_dir):
        shutil.rmtree(models_dir)
    os.makedirs(models_dir)
    
    # Copy Nemotron TensorRT-LLM engine directory if it exists locally
    nemotron_path = tensorrt_models.get("nemotron_tensorrt_llm")
    if nemotron_path and os.path.exists(nemotron_path):
        if os.path.isdir(nemotron_path):
            # Copy entire Nemotron TensorRT engine directory
            target_dir = os.path.join(models_dir, "nemotron_tensorrt_llm_engine")
            shutil.copytree(nemotron_path, target_dir)
            logger.info(f"📦 Copied Nemotron TensorRT-LLM engine directory: {nemotron_path} -> {target_dir}")
        else:
            # Copy single file
            target_filename = "nemotron_tensorrt_llm_engine.bin"
            shutil.copy2(nemotron_path, os.path.join(models_dir, target_filename))
            logger.info(f"📦 Copied Nemotron TensorRT-LLM engine file: {target_filename}")
    else:
        # If path doesn't exist locally, it's a HuggingFace repo ID
        logger.info(f"☁️ Nemotron model is HuggingFace repo ID: {nemotron_path}")
        logger.info("📥 Model reference will be stored in metadata for on-demand loading")
    
    # Copy ONNX models if they exist
    onnx_count = 0
    if os.path.exists(onnx_dir):
        for onnx_file in os.listdir(onnx_dir):
            if onnx_file.endswith('.onnx'):
                src_path = os.path.join(onnx_dir, onnx_file)
                dst_path = os.path.join(models_dir, onnx_file)
                shutil.copy2(src_path, dst_path)
                onnx_count += 1
                logger.info(f"📦 Copied ONNX model: {onnx_file}")
    
    # Create metadata file with Nemotron model info
    metadata = {
        "models": {
            "nemotron_tensorrt_llm": tensorrt_models.get("nemotron_tensorrt_llm"),
            "model_type": "nvidia/Llama-3.1-Nemotron-Nano-8B-v1",
            "embedding_model": tensorrt_models.get("embedding_model")
        },
        "onnx_available": onnx_count > 0,
        "onnx_count": onnx_count,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    }
    
    metadata_path = os.path.join(models_dir, "models_metadata.json")
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    logger.info(f"📋 Created models metadata with Nemotron info: {metadata_path}")
    
    return models_dir

logger.info("✅ Helper functions defined successfully (includes Nemotron TensorRT-LLM handling)")

2025-12-04 16:41:37 - INFO - ✅ Helper functions defined successfully (includes Nemotron TensorRT-LLM handling)


In [11]:
%%time

# 2. Start an MLflow run and log + register the model using new architecture
with mlflow.start_run(run_name=RUN_NAME) as run:
    print(f"Started MLflow run: {run.info.run_id}")
    tensorrt_models, loaded_models = load_models_for_onnx_conversion()
    onnx_dir = create_and_convert_onnx_models(loaded_models)
    model_dir = prepare_models_with_onnx(tensorrt_models, onnx_dir)

    # Use the new Logger class for model registration with signature
    # Explicitly disable ONNX export to avoid parameter conflicts
    Logger.log_model(
        signature=signature, 
        artifact_path=MODEL_NAME, 
        config_path="../configs/config.yaml",
        docs_path="../data/",
        model_path=model_dir,
        demo_folder="../demo"
    )

    model_uri = f"runs:/{run.info.run_id}/{MODEL_NAME}"
    mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)

# ------------------------- Success Confirmation -------------------------

print(f"✅ Model '{MODEL_NAME}' successfully logged and registered under experiment '{EXPERIMENT_NAME}'.")

2025-12-04 16:41:38 - INFO - Nemotron model configured as HuggingFace repo: nvidia/Llama-3.1-Nemotron-Nano-8B-v1
2025-12-04 16:41:38 - INFO - Model will be downloaded on-demand during inference


Started MLflow run: ea628e04081847ae9cf87a3a49af9926


2025-12-04 16:41:49 - INFO - Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2
2025-12-04 16:41:49 - INFO - All models loaded into memory for ONNX conversion
2025-12-04 16:41:50 - INFO - ✅ Converted embedding model to ONNX: /tmp/onnx_models_agentic_rag/embedding_model.onnx
2025-12-04 16:41:50 - INFO - 📁 ONNX models saved to directory: /tmp/onnx_models_agentic_rag
2025-12-04 16:41:50 - INFO - ☁️ Nemotron model is HuggingFace repo ID: nvidia/Llama-3.1-Nemotron-Nano-8B-v1
2025-12-04 16:41:50 - INFO - 📥 Model reference will be stored in metadata for on-demand loading
2025-12-04 16:41:50 - INFO - 📦 Copied ONNX model: embedding_model.onnx
2025-12-04 16:41:50 - INFO - 📋 Created models metadata with Nemotron info: /tmp/all_agentic_rag_models/models_metadata.json
/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Chec

[TensorRT-LLM] TensorRT-LLM version: 0.18.0
[TensorRT-LLM][INFO] Engine version 0.18.0 found in the config file, assuming engine(s) built by new builder API.
[TensorRT-LLM][INFO] Refreshed the MPI local session
[TensorRT-LLM][INFO] MPI size: 1, MPI local size: 1, rank: 0
[TensorRT-LLM][INFO] Rank 0 is using GPU 0
[TensorRT-LLM][WARNING] Fix optionalParams : KV cache reuse disabled because model was not built with paged context FMHA support
[TensorRT-LLM][INFO] TRTGptModel maxNumSequences: 2048
[TensorRT-LLM][INFO] TRTGptModel maxBatchSize: 2048
[TensorRT-LLM][INFO] TRTGptModel maxBeamWidth: 1
[TensorRT-LLM][INFO] TRTGptModel maxSequenceLen: 131072
[TensorRT-LLM][INFO] TRTGptModel maxDraftLen: 0
[TensorRT-LLM][INFO] TRTGptModel mMaxAttentionWindowSize: (131072) * 32
[TensorRT-LLM][INFO] TRTGptModel enableTrtOverlap: 0
[TensorRT-LLM][INFO] TRTGptModel normalizeLogProbs: 0
[TensorRT-LLM][INFO] TRTGptModel maxNumTokens: 8192
[TensorRT-LLM][INFO] TRTGptModel maxInputLen: 8192 = min(maxSeque

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.36it/s]
2025-12-04 16:47:40 - INFO - 🔧 Generating ONNX model(s) for specified models...
2025-12-04 16:47:40 - INFO - 🔄 Starting conversion for model: embedding_model
2025-12-04 16:47:40 - INFO - 🔄 Converting transformers model: embedding_model
2025-12-04 16:47:41 - INFO - 📁 Model directory: embedding_model
2025-12-04 16:47:41 - INFO - 🔍 Model identified as: transformers
2025-12-04 16:47:41 - INFO - 🤗 Converting loaded Transformers model for task: feature-extraction with opset 17
2025-12-04 16:47:50 - INFO - ✅ Transformers model exported to: embedding_model/model.onnx
2025-12-04 16:47:50 - INFO - ✅ Converted embedding_model to directory: embedding_model
2025-12-04 16:47:50 - INFO - 🔄 Starting conversion for model: nemotron_model
2025-12-04 16:47:50 - INFO - 🔄 Converting pytorch model: nemotron_model
2025-12-04 16:47:50 - INFO - 📁 Model directory: nemotron_model
2025-12-04 16:47:50 - INFO - 🔍 Model identified as: pytorch
20

✅ Model 'Agentic_RAG_Model' successfully logged and registered under experiment 'Agentic_RAG_Experiment'.
CPU times: user 5min 41s, sys: 2min 47s, total: 8min 28s
Wall time: 11min 41s


In [12]:
# 3. Retrieve the latest version from the Model Registry
client = MlflowClient()
versions = client.get_latest_versions(MODEL_NAME, stages=["None"])
if not versions:
    raise RuntimeError(f"No registered versions found for model '{MODEL_NAME}'.")
latest_version = versions[0].version

model_info = mlflow.models.get_model_info(f"models:/{MODEL_NAME}/{latest_version}")
print(f"Latest registered version of '{MODEL_NAME}': {latest_version}")
print(f"Signature: {model_info.signature}")

Latest registered version of 'Agentic_RAG_Model': 11
Signature: inputs: 
  ['query': string (required)]
outputs: 
  ['answer': string (required), 'retrieved_chunks': string (required), 'messages': string (required)]
params: 
  None



# Log Results to MLFlow

In [13]:
%%time

# 4. Load the model from the Model Registry
loaded_model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{latest_version}")
print(f"Successfully loaded model '{MODEL_NAME}' version {latest_version} for inference.")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Loading Model: [1/3]	Downloading HF model
Downloaded model to /root/.cache/huggingface/hub/models--nvidia--Llama-3.1-Nemotron-Nano-8B-v1/snapshots/54641c1611fcff44fa4865626462445e0a153fc7
Time: 4.897s
Loading Model: [2/3]	Loading HF model to memory
230it [00:01, 199.07it/s]
Time: 1.839s
Loading Model: [3/3]	Building TRT-LLM engine
Time: 291.669s
Loading model done.
Total latency: 298.406s


[TensorRT-LLM] TensorRT-LLM version: 0.18.0
[TensorRT-LLM][INFO] Engine version 0.18.0 found in the config file, assuming engine(s) built by new builder API.
[TensorRT-LLM][INFO] Refreshed the MPI local session
[TensorRT-LLM][INFO] MPI size: 1, MPI local size: 1, rank: 0
[TensorRT-LLM][INFO] Rank 0 is using GPU 0
[TensorRT-LLM][WARNING] Fix optionalParams : KV cache reuse disabled because model was not built with paged context FMHA support
[TensorRT-LLM][INFO] TRTGptModel maxNumSequences: 2048
[TensorRT-LLM][INFO] TRTGptModel maxBatchSize: 2048
[TensorRT-LLM][INFO] TRTGptModel maxBeamWidth: 1
[TensorRT-LLM][INFO] TRTGptModel maxSequenceLen: 131072
[TensorRT-LLM][INFO] TRTGptModel maxDraftLen: 0
[TensorRT-LLM][INFO] TRTGptModel mMaxAttentionWindowSize: (131072) * 32
[TensorRT-LLM][INFO] TRTGptModel enableTrtOverlap: 0
[TensorRT-LLM][INFO] TRTGptModel normalizeLogProbs: 0
[TensorRT-LLM][INFO] TRTGptModel maxNumTokens: 8192
[TensorRT-LLM][INFO] TRTGptModel maxInputLen: 8192 = min(maxSeque

In [14]:
# 5. Run a sample inference using the loaded model
sample_query = "What is the hardware requirement for AI Studio?"
input_payload = {"query": sample_query}

print("\n=== Running Sample Inference ===")
result = loaded_model.predict(input_payload)


=== Running Sample Inference ===


Processed requests:   0%|          | 0/1 [00:00<?, ?it/s]

[TensorRT-LLM][WARNING] Prompt length + number of requested output tokens + draft tokens per step (103 + 32 + 0) exceeds maximum sequence length (128). Number of requested output tokens is changed to (25).


Processed requests: 100%|██████████| 1/1 [00:12<00:00, 12.39s/it]


In [15]:
# 6. Print results
print(f"Query:")
print("{sample_query}\n")
print("\n==============\n")

print("Answer:")
print(result.get("answer", "<no answer>"), "\n")
print("\n==============\n")

print("Retrieved Chunks:")
for idx, chunk in enumerate(result.get("retrieved_chunks", []), start=1):
    print(f"  {idx}. {chunk[:100]}{'...' if len(chunk)>100 else ''}")

print("\n==============\n")
print("\nMessage History:")
for msg in result.get("messages", []):
    role = msg.get("role", "<unknown>")
    content = msg.get("content", "")
    print(f"  [{role}]: {content}")

Query:
{sample_query}



Answer:
AMD Ryzen™ 9 processor, Intel Core™ i5 12th generation processor, or higher 



Retrieved Chunks:



Message History:
  [user]: What is the hardware requirement for AI Studio?
  [developer]: Relevance check result:
  [assistant]: Yes


In [16]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")

2025-12-04 17:01:16 - INFO - ⏱️ Total execution time: 20m 53.15s
2025-12-04 17:01:16 - INFO - ✅ Notebook execution completed successfully.


In [1]:
status = "Notebook execution completed successfully"
print(f"Message: {status}")

Message: Notebook execution completed successfully


In [ ]:
app = get_ipython()
app.kernel.do_shutdown(restart=False)

Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).